## **Web assistant via `RAG` Powered**

- llamaParser for reading documents
- hydride search system
- advanced re-ranker algo
- self corrective powered rag
- zero hallucinations

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load Environment Variables
load_dotenv()

# Verify Key
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("Enter OpenAI API Key: ")

print("Environment Ready.")

Environment Ready.


## **Load the data**

In [3]:
import json
import pandas as pd
import os

# Define Paths
catalog_path = "data/product_catalog.json"
faq_path = "data/faq.csv"
logic_path = "data/business_logic.csv"
policy_path = "data/policy.txt"

# 1. Load Product Catalog
if os.path.exists(catalog_path):
    with open(catalog_path, "r") as f:
        product_data = json.load(f)
    print(f"✅ Loaded Product Catalog: {len(product_data)} items")
else:
    print(f"❌ Error: {catalog_path} not found.")

# 2. Load FAQ (General)
if os.path.exists(faq_path):
    faq_data = pd.read_csv(faq_path)
    print(f"✅ Loaded FAQ: {len(faq_data)} rows")
else:
    print(f"❌ Error: {faq_path} not found.")

# 3. Load Business Logic (Deterministic)
if os.path.exists(logic_path):
    logic_data = pd.read_csv(logic_path)
    print(f"✅ Loaded Business Logic: {len(logic_data)} rows")
else:
    print(f"❌ Error: {logic_path} not found.")

# 4. Load Policy
if os.path.exists(policy_path):
    with open(policy_path, "r") as f:
        policy_text = f.read()
    print(f"✅ Loaded Policy: {len(policy_text)} chars")
else:
    print(f"❌ Error: {policy_path} not found.")

✅ Loaded Product Catalog: 5 items
✅ Loaded FAQ: 17 rows
✅ Loaded Business Logic: 9 rows
✅ Loaded Policy: 2449 chars


## **Process the data**

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_real_data_for_indexing():
    documents = []

    # --- 1. JSON: Product Catalog (HQI Strategy) ---
    # We embed the QUESTIONS, but return the CONTENT.
    for item in product_data:
        for q in item['indexing_questions']:
            doc = Document(
                page_content=q,  # <--- Embed the Question
                metadata={
                    "source": "product_catalog",
                    "answer_content": item['content'], # <--- LLM reads this
                    "url_context": item.get('url_context', '/'),
                    "category": item['category'],
                    "type": "technical"
                }
            )
            documents.append(doc)

    # --- 2. CSV: FAQ (Standard Match) ---
    for _, row in faq_data.iterrows():
        doc = Document(
            page_content=row['question'],
            metadata={
                "source": "faq",
                "answer_content": row['answer'],
                "category": row['category'],
                "type": "general_faq"
            }
        )
        documents.append(doc)

    # --- 3. CSV: Business Logic (High Priority Match) ---
    # We treat these same as FAQ but tag them 'business_logic'
    for _, row in logic_data.iterrows():
        doc = Document(
            page_content=row['question'],
            metadata={
                "source": "business_logic",
                "answer_content": row['answer'],
                "category": row['category'],
                "type": "deterministic"
            }
        )
        documents.append(doc)

    # --- 4. TXT: Policy (Chunking) ---
    # Policy is long text, so we chunk it using standard splitting.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    policy_chunks = text_splitter.create_documents([policy_text])
    
    for chunk in policy_chunks:
        chunk.metadata["source"] = "policy"
        chunk.metadata["type"] = "legal"
        # For policy, the answer is the chunk itself
        chunk.metadata["answer_content"] = chunk.page_content 
        documents.append(chunk)

    return documents

# Execute Processing
raw_docs = process_real_data_for_indexing()
print(f"🎉 Processed {len(raw_docs)} total vectors for indexing.")
print(f"Sample Metadata: {raw_docs[0].metadata}")

🎉 Processed 62 total vectors for indexing.
Sample Metadata: {'source': 'product_catalog', 'answer_content': 'BYV architects high-performance digital platforms using Next.js and Cloud-Native technologies. Unlike agencies that rely on brittle legacy systems (like WordPress), we build API-first architectures using React, FastAPI, and PostgreSQL. Key features include sub-100ms load times via Edge Caching, Enterprise Security (SOC-2 Ready, 256-bit encryption), and Mobile Native deployment (iOS/Android from a single codebase). This infrastructure is designed for scalability, automatically handling traffic spikes without downtime.', 'url_context': '/products/infrastructure', 'category': 'infrastructure', 'type': 'technical'}


## **Build the `Vector Store`**

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build Index
print("Building Vector Index... (This simulates ingestion)")
vector_store = FAISS.from_documents(documents=raw_docs, embedding=embeddings)

# Save locally (Optional, simulates production persistence)
vector_store.save_local("faiss_index_store")
print("Index saved to disk.")

Building Vector Index... (This simulates ingestion)
Index saved to disk.


## **Configure the Retriever**

In [6]:
# Load from disk
loaded_vector_store = FAISS.load_local(
    "faiss_index_store", 
    embeddings, 
    allow_dangerous_deserialization=True # Safe since we made it
)

# Create Retriever
# k=3 means we get the top 3 matches
retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 3})

print("Retriever Ready.")

Retriever Ready.


## **Test case 1**

In [7]:
query = "Can the bot read my invoices?"

results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} (Score: Match via '{doc.page_content}') ---")
    print(f"Retrieved Answer: {doc.metadata['answer_content']}")
    print(f"Source: {doc.metadata['source']}")
    print("\n")

Query: Can the bot read my invoices?

--- Result 1 (Score: Match via 'Can the AI read invoices or images?') ---
Retrieved Answer: BYV's Autonomous Agents are the 'Digital Workforce' that automates complex business operations. Powered by LangGraph event-driven architecture, these agents plan, reason, and execute tasks 24/7. Use cases include: Sales Automation (negotiating via chat and logging deals to CRM), Operations (tracking factory inventory and predicting supply chain delays), and Document Processing (using Vision to read invoices). The system includes 'Human-in-the-Loop' protocols, where agents ask for manager approval if confidence is low.
Source: product_catalog


--- Result 2 (Score: Match via 'Can the AI send emails or update my CRM?') ---
Retrieved Answer: BYV's Autonomous Agents are the 'Digital Workforce' that automates complex business operations. Powered by LangGraph event-driven architecture, these agents plan, reason, and execute tasks 24/7. Use cases include: Sales Aut

## **Test case 2**

In [8]:
query = "How much does it cost to build an agent?"

results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Retrieved Answer: {doc.metadata['answer_content']}")
    print(f"Category: {doc.metadata['category']}")
    print("\n")

Query: How much does it cost to build an agent?

--- Result 1 ---
Retrieved Answer: Simple RAG chatbots start around $2,500. Complex Autonomous Agents (that integrate with CRM/Email) typically range from $5,000 to $15,000 depending on the complexity of the logic and data integration.
Category: pricing


--- Result 2 ---
Retrieved Answer: AI implementations generally start at $2,500 USD for a standard RAG knowledge base setup. Complex Autonomous Agents or Private Fine-tuned models typically range from $5,000 to $15,000+ USD depending on data volume and logic complexity.
Category: pricing


--- Result 3 ---
Retrieved Answer: BYV offers Custom AI Model Fine-Tuning services. We train open-source foundation models (like Llama-3, Mistral, or Falcon) specifically on your proprietary enterprise data. This ensures Data Sovereignty—your data never leaves your Private Cloud (VPC) or On-Premise servers, unlike public APIs like GPT-4. These specialized models are 90% cheaper to run at scale and are

## **Test case 3**

In [9]:
# This simulates the context string passed to Llama-3
def format_docs(docs):
    return "\n\n".join(
        [f"Source ({d.metadata['source']}): {d.metadata['answer_content']}" for d in docs]
    )

query = "tell me about RAG security"
results = retriever.invoke(query)
context_block = format_docs(results)

print("--- SYSTEM PROMPT CONTEXT BLOCK ---")
print(context_block)

--- SYSTEM PROMPT CONTEXT BLOCK ---
Source (product_catalog): BYV's Enterprise RAG (Retrieval Augmented Generation) engine allows your team to chat with your internal knowledge base. We ingest unstructured data (PDFs, SharePoint, Google Drive) and structured data (SQL) into a Vector Store. The system provides 'Grounded Truth'—every answer includes citations linking back to the source document, eliminating hallucinations. It supports Role-Based Access Control (RBAC), ensuring users only access data they are permitted to see.

Source (product_catalog): BYV offers Custom AI Model Fine-Tuning services. We train open-source foundation models (like Llama-3, Mistral, or Falcon) specifically on your proprietary enterprise data. This ensures Data Sovereignty—your data never leaves your Private Cloud (VPC) or On-Premise servers, unlike public APIs like GPT-4. These specialized models are 90% cheaper to run at scale and are hyper-specialized for your industry jargon, whether it is Garment Manufac

In [10]:
from langchain_community.retrievers import BM25Retriever
# from langchain.retrievers import EnsembleRetriever


# 1. Setup FAISS (Semantic)
faiss_retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 3})

# 2. Setup BM25 (Keyword) - Use the same 'raw_docs' list from earlier
bm25_retriever = BM25Retriever.from_documents(raw_docs)
bm25_retriever.k = 3

# 3. Combine (Hybrid)
# Weights: 0.5 for Semantic, 0.5 for Keyword. This is standard.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.5, 0.5]
# )

# print("Hybrid Retriever (Ensemble) Ready.")

In [14]:
from langchain_community.retrievers import BM25Retriever

# Setup retrievers
faiss_retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(raw_docs)
bm25_retriever.k = 10

def rrf_hybrid_search(query, k=3, rrf_k=60):
    """
    Reciprocal Rank Fusion - Industry standard for hybrid search
    """
    # Use invoke() instead of get_relevant_documents()
    bm25_docs = bm25_retriever.invoke(query)
    faiss_docs = faiss_retriever.invoke(query)
    
    # Calculate RRF scores
    doc_scores = {}
    
    # BM25 RRF scores
    for rank, doc in enumerate(bm25_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    # FAISS RRF scores
    for rank, doc in enumerate(faiss_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    # Sort and return
    sorted_docs = sorted(doc_scores.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in sorted_docs[:k]]

# Test it with your actual query
results = rrf_hybrid_search("tell me about RAG security", k=5)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])

print("\n✅ RRF Hybrid Retriever Ready!")


--- Result 1 ---
How does RAG work?

--- Result 2 ---
What is the price for an AI Agent or RAG system?

--- Result 3 ---
3. AI SAFETY & ETHICS
- Zero Retention Policy (API): When using third-party LLM APIs (like OpenAI Enterprise), we configure policies to ensure zero data retention for model training purposes.
- Hallucination Control: Our RAG systems are engineered with "Groundedness Checks." If the system cannot fin

--- Result 4 ---
Is the knowledge base secure?

--- Result 5 ---
What is the difference between fine-tuning and RAG?

✅ RRF Hybrid Retriever Ready!


In [15]:
"""
COPY-PASTE THIS INTO YOUR JUPYTER NOTEBOOK
Run this after you have loaded_vector_store and raw_docs ready
"""

from langchain_community.retrievers import BM25Retriever

# Setup retrievers
faiss_retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(raw_docs)
bm25_retriever.k = 10

# RRF Hybrid Search Function
def rrf_hybrid_search(query, k=3, rrf_k=60):
    bm25_docs = bm25_retriever.invoke(query)
    faiss_docs = faiss_retriever.invoke(query)
    
    doc_scores = {}
    
    for rank, doc in enumerate(bm25_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    for rank, doc in enumerate(faiss_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    sorted_docs = sorted(doc_scores.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in sorted_docs[:k]]

# Quick test
query = "What technology stack does BYV use?"
results = rrf_hybrid_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"  {doc.page_content[:150]}...")
    print()

Query: What technology stack does BYV use?

Result 1:
  Where is BYV located?...

Result 2:
  What technology stack does BYV use?...

Result 3:
  What makes BYV different from other agencies?...



In [16]:
"""
Quick Test for Hybrid Retriever (RRF)
Assumes you already have:
- loaded_vector_store (your FAISS vectorstore)
- raw_docs (your original documents)
"""

from langchain_community.retrievers import BM25Retriever

# ============================================================================
# SETUP RETRIEVERS (assuming you already have loaded_vector_store and raw_docs)
# ============================================================================

# 1. FAISS Retriever (from your existing vectorstore)
faiss_retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 10})

# 2. BM25 Retriever (from your raw documents)
bm25_retriever = BM25Retriever.from_documents(raw_docs)
bm25_retriever.k = 10

# ============================================================================
# RRF HYBRID SEARCH FUNCTION
# ============================================================================

def rrf_hybrid_search(query, k=3, rrf_k=60):
    """
    Reciprocal Rank Fusion - Combines BM25 + FAISS
    """
    # Get results from both retrievers
    bm25_docs = bm25_retriever.invoke(query)
    faiss_docs = faiss_retriever.invoke(query)
    
    # Calculate RRF scores
    doc_scores = {}
    
    # BM25 scores
    for rank, doc in enumerate(bm25_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    # FAISS scores
    for rank, doc in enumerate(faiss_docs, 1):
        content = doc.page_content
        score = 1 / (rrf_k + rank)
        if content in doc_scores:
            doc_scores[content]['score'] += score
        else:
            doc_scores[content] = {'doc': doc, 'score': score}
    
    # Sort and return top k
    sorted_docs = sorted(doc_scores.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in sorted_docs[:k]]

print("✅ Hybrid Retriever Ready!\n")

# ============================================================================
# TEST QUERIES FOR YOUR BYV DOCUMENTS
# ============================================================================

# Test Query 1: Technology Stack (should find Digital Infrastructure)
print("="*80)
print("🔍 TEST 1: What technology stack does BYV use?")
print("="*80)
results = rrf_hybrid_search("What technology stack does BYV use?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 2: Data Privacy (should find Custom AI Models + Privacy Policy)
print("="*80)
print("🔍 TEST 2: Is my data safe and private?")
print("="*80)
results = rrf_hybrid_search("Is my data safe and private?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 3: Custom AI Training (should find Custom AI Models)
print("="*80)
print("🔍 TEST 3: Can you train an AI on my private data?")
print("="*80)
results = rrf_hybrid_search("Can you train an AI on my private data?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 4: Mobile Apps (should find Digital Infrastructure)
print("="*80)
print("🔍 TEST 4: Do you build mobile apps?")
print("="*80)
results = rrf_hybrid_search("Do you build mobile apps?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 5: Agents 24/7 (should find Autonomous Agents)
print("="*80)
print("🔍 TEST 5: Do agents work 24/7?")
print("="*80)
results = rrf_hybrid_search("Do agents work 24/7?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 6: GDPR Compliance (should find Privacy Policy)
print("="*80)
print("🔍 TEST 6: What is GDPR compliance at BYV?")
print("="*80)
results = rrf_hybrid_search("What is GDPR compliance at BYV?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 7: Security/Encryption (should find Privacy Policy + Infrastructure)
print("="*80)
print("🔍 TEST 7: How does BYV handle encryption?")
print("="*80)
results = rrf_hybrid_search("How does BYV handle encryption?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

# Test Query 8: RAG System (should find Enterprise RAG)
print("="*80)
print("🔍 TEST 8: How does RAG work?")
print("="*80)
results = rrf_hybrid_search("How does RAG work?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content: {doc.page_content[:200]}...")
print("\n")

print("="*80)
print("✅ ALL TESTS COMPLETED!")
print("="*80)
print("\n💡 You can now use rrf_hybrid_search() in your RAG pipeline")
print("   Example: results = rrf_hybrid_search('your query here', k=5)")

✅ Hybrid Retriever Ready!

🔍 TEST 1: What technology stack does BYV use?

📄 Result 1:
   Metadata: {'source': 'faq', 'answer_content': 'BYV is headquartered in Dhaka, Bangladesh (Bashundhara R/A). However, we operate as a cloud-first engineering collective, serving clients across North America, Europe, and Asia with 24/7 overlap capabilities.', 'category': 'company', 'type': 'general_faq'}
   Content: Where is BYV located?...

📄 Result 2:
   Metadata: {'source': 'product_catalog', 'answer_content': 'BYV architects high-performance digital platforms using Next.js and Cloud-Native technologies. Unlike agencies that rely on brittle legacy systems (like WordPress), we build API-first architectures using React, FastAPI, and PostgreSQL. Key features include sub-100ms load times via Edge Caching, Enterprise Security (SOC-2 Ready, 256-bit encryption), and Mobile Native deployment (iOS/Android from a single codebase). This infrastructure is designed for scalability, automatically handling traff